## The Need for Convolutional Neural Networks

This notebook explains why Convolutional Neural Networks (CNNs) are used for image
data. We first train a plain fully connected network, referred to here as a DNN, on a
real image classification problem and observe its limitations. We then build a CNN
and show how convolution addresses each limitation.

We use `FashionMNIST`, a dataset of 70,000 grayscale photographs of clothing items
(28x28 pixels, 10 classes: `t-shirt, trouser, pullover, dress, coat, sandal, shirt,
sneaker, bag, and ankle boot`).

It is a real classification task, and it is small
enough to train on a laptop CPU in a few minutes.

We study two limitations of a plain DNN on images. Both are measured empirically
below.

1. Parameter explosion. A fully connected layer needs one weight for every pixel and
   hidden unit pair. The number of weights grows with the number of pixels in the
   image.
2. No translation invariance. A DNN's weights are tied to fixed pixel positions. If
   the same object appears a few pixels to the side, the network can fail badly.

A CNN addresses both problems using two ideas: local receptive fields and weight
sharing, where the same small filter is applied at every position in the image.
Pooling adds a further degree of tolerance to small shifts.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cpu")
print("device:", device)

### 1. The dataset: classifying clothing photographs (FashionMNIST)

FashionMNIST is a harder, drop in replacement for the classic MNIST digit dataset. 
- It has the same size and format: 28x28 grayscale images, 
    - 10 classes
    - 60,000 training images and 
    - 10,000 test images. 
  
Its classes, however, are visually similar to each other 
- shirts, pullovers, and coats, or sneakers, sandals, and ankle boots.

This makes it a better test of whether a model can learn real shapes 
and textures, rather than simple outlines.

Source: Fashion-MNIST is created and published by Zalando Research. The original
repository, with full details and a citation, is available at
https://github.com/zalandoresearch/fashion-mnist. The raw files are downloaded
directly from the project's hosted mirror in the next cell.

In [ ]:
# Download the raw Fashion-MNIST archives directly from the source mirror into data/,
# before loading them with torchvision. wget's -nc (no-clobber) flag skips the download
# if a file is already present, so re-running this cell will not re-download the data.
DATA_URL = "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com"
RAW_DIR = "data/FashionMNIST/raw"

!mkdir -p {RAW_DIR}
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/train-images-idx3-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/train-labels-idx1-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/t10k-images-idx3-ubyte.gz
!wget -nc -nv -P {RAW_DIR} {DATA_URL}/t10k-labels-idx1-ubyte.gz
!ls -lh {RAW_DIR}

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

transform = transforms.ToTensor()  # scales pixels to [0, 1], shape (1, 28, 28)

In [ ]:
# download=True only extracts here: the archives were already fetched by wget above
train_ds = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

print(f"train images: {len(train_ds)}, test images: {len(test_ds)}, image shape: {train_ds[0][0].shape}")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axes.flat):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
fig.suptitle("Sample FashionMNIST images")
plt.tight_layout()
plt.show()

### 2. Limitation one: flattening discards spatial structure

A plain DNN can only process a flat vector. So the first step with any image is to
`flatten` the 2D grid of pixels into a 1D list. Each pixel then receives its own
independent weight to every hidden unit. The network has no notion that pixel (5, 5)
is next to pixel (5, 6). In addition, the number of weights in this first layer grows
directly with the number of pixels in the image.

In [ ]:
def fc_layer_params(in_features, hidden_units):
    return in_features * hidden_units + hidden_units  # weights + biases

# Our FashionMNIST images: 28x28 grayscale
mnist_params = fc_layer_params(28 * 28, 256)
print(f"28x28 grayscale image -> first FC layer (256 units): {mnist_params:,} weights")

# A modest real photo (small by phone-camera standards): 224x224 RGB
photo_params = fc_layer_params(224 * 224 * 3, 256)
print(f"224x224 RGB photo    -> first FC layer (256 units): {photo_params:,} weights")
print(f"({photo_params / mnist_params:.0f}x more weights just from a more realistic image size)")

# A single 3x3 convolution kernel doesn't care about image size at all:
conv_params = 3 * 3 * 1 * 16 + 16  # 3x3 kernel, 1 input channel, 16 filters
print(f"\n3x3 conv layer (16 filters), ANY image size: {conv_params:,} weights")

A fully connected layer's cost grows with the size of the image. A convolution
kernel's cost depends only on the size of the kernel. This is the first structural
problem that CNNs solve. Before fixing it, let us confirm the DNN's limitation with an
experiment.

### 3. Baseline: a plain DNN (MLP) on FashionMNIST

We use a standard two hidden layer MLP: flatten, then 256 units, then 128 units, then
10 output classes.

In [ ]:
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


dnn = DNN().to(device)
print(dnn)
print(f"\nDNN parameters: {count_params(dnn):,}")

In [ ]:
def train_model(model, loader, epochs=5, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"loss": [], "acc": []}
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)

        epoch_loss, epoch_acc = running_loss / total, correct / total
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)
        print(f"  epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}  train_acc={epoch_acc:.4f}")
    return history


@torch.no_grad()
def evaluate(model, loader, shift=None):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if shift is not None:
            xb = shift_batch(xb, *shift)
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

In [ ]:
print("Training DNN...")
dnn_history = train_model(dnn, train_loader, epochs=5)
dnn_test_acc = evaluate(dnn, test_loader)
print(f"\nDNN test accuracy: {dnn_test_acc:.4f}")

### 4. Limitation two: DNNs are not robust to translation

In a real photograph, an object rarely appears at the exact same pixel location every
time. A useful vision model should recognize a sneaker whether it is centered in the
frame or shifted slightly to one side. However, a DNN's first layer weights are tied
to fixed pixel positions. The weight connected to pixel (5, 5) has no relationship to
the weight connected to pixel (5, 8). When the image is shifted, every pixel lands on
a weight that was never trained to interpret it.

We now shift each test image by a few pixels, filling the revealed background with
zeros, as if the object had physically moved. We then measure accuracy using the same
trained DNN, without any retraining.

In [ ]:
def shift_batch(x, dx, dy):
    # shift image content by (dx, dy) pixels; revealed area is filled with zeros
    x = F.pad(x, (4, 4, 4, 4))
    x = torch.roll(x, shifts=(dy, dx), dims=(2, 3))
    return x[:, :, 4:32, 4:32]


# Visualize the effect of a shift
img, label = test_ds[0]
img = img.unsqueeze(0)
shifted = shift_batch(img, dx=4, dy=4)

fig, axes = plt.subplots(1, 2, figsize=(5, 2.7))
axes[0].imshow(img.squeeze(), cmap="gray"); axes[0].set_title("original"); axes[0].axis("off")
axes[1].imshow(shifted.squeeze(), cmap="gray"); axes[1].set_title("shifted by (4, 4) px"); axes[1].axis("off")
fig.suptitle(f"Same {class_names[label]} — just moved a few pixels")
plt.tight_layout()
plt.show()

In [ ]:
shifts = [(0, 0), (2, 2), (4, 4), (6, 6)]
dnn_shift_acc = [evaluate(dnn, test_loader, shift=s) for s in shifts]

for s, acc in zip(shifts, dnn_shift_acc):
    print(f"  shift={s}:  DNN accuracy = {acc:.4f}")

The network, its weights, and the clothing item are unchanged. Only the position has
moved by a few pixels, yet accuracy collapses. The DNN did not learn to detect a
sneaker's shape. It learned to detect specific pixel intensities at specific pixel
positions.

### 5. The CNN solution: local receptive fields, weight sharing, and pooling

A convolutional layer slides a small kernel (for example, 3x3) across the image and
reuses the same weights at every position.

- Local receptive field: each output value depends only on a small neighborhood of
  pixels, so the network can learn spatially local patterns such as edges, corners,
  and textures.
- Weight sharing: because the same kernel is applied everywhere, a feature learned in
  one part of the image is automatically detected everywhere else, including after a
  small shift.
- Pooling (for example, max pooling) summarizes small neighborhoods into a single
  value, adding further tolerance to small shifts.

We use the same training setup as the DNN: the same optimizer, the same number of
epochs, and the same data. This keeps the comparison fair.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14 -> 7x7
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.head(self.conv(x))


cnn = CNN().to(device)
print(cnn)
print(f"\nCNN parameters: {count_params(cnn):,}  (DNN had {count_params(dnn):,})")

In [ ]:
print("Training CNN...")
cnn_history = train_model(cnn, train_loader, epochs=5)
cnn_test_acc = evaluate(cnn, test_loader)
print(f"\nCNN test accuracy: {cnn_test_acc:.4f}")

In [ ]:
cnn_shift_acc = [evaluate(cnn, test_loader, shift=s) for s in shifts]

print(f"{'shift':<10}{'DNN acc':<12}{'CNN acc':<12}")
for s, a_dnn, a_cnn in zip(shifts, dnn_shift_acc, cnn_shift_acc):
    print(f"{str(s):<10}{a_dnn:<12.4f}{a_cnn:<12.4f}")

In [ ]:
x = np.arange(len(shifts))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width / 2, dnn_shift_acc, width, label="DNN")
ax.bar(x + width / 2, cnn_shift_acc, width, label="CNN")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in shifts])
ax.set_xlabel("pixel shift (dx, dy)")
ax.set_ylabel("test accuracy")
ax.set_title("Robustness to translation: DNN vs CNN\n(same models trained above, no retraining on shifted data)")
ax.legend()
plt.tight_layout()
plt.show()

The CNN retains noticeably more accuracy as the shift increases. It is not perfectly
shift invariant, since the flatten and fully connected head at the end still depends
on the position of each feature. However, the convolutional feature extractor itself
does not need to relearn what a sneaker looks like just because it moved a few
pixels.

### 6. Extension: global average pooling for stronger translation invariance

The remaining shift sensitivity above comes from the final flatten and linear head,
which still depends on the position of each feature. Replacing this head with global
average pooling, which averages every feature map to a single number before
classification, removes position information entirely. This reduces accuracy, since
the classifier now has less information to work with. Robustness to shifts, however,
improves substantially. This shows that translation invariance is a direct and
controllable consequence of the architecture, not something a CNN receives
automatically just by using convolution.

In [ ]:
class CNN_GAP(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14 -> 7x7
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # 7x7x64 -> 1x1x64, throws away position
        )
        self.head = nn.Linear(64, 10)

    def forward(self, x):
        x = self.conv(x).flatten(1)
        return self.head(x)


cnn_gap = CNN_GAP().to(device)
print(f"CNN_GAP parameters: {count_params(cnn_gap):,}")

In [ ]:
print("Training CNN_GAP...")
cnn_gap_history = train_model(cnn_gap, train_loader, epochs=5)
cnn_gap_test_acc = evaluate(cnn_gap, test_loader)
print(f"\nCNN_GAP test accuracy: {evaluate(cnn_gap, test_loader):.4f}")

In [ ]:

cnn_gap_shift_acc = [evaluate(cnn_gap, test_loader, shift=s) for s in shifts]

print(f"\nCNN_GAP test accuracy: {cnn_gap_test_acc:.4f}")
for s, acc in zip(shifts, cnn_gap_shift_acc):
    print(f"  shift={s}:  CNN_GAP accuracy = {acc:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(x, dnn_shift_acc, "o-", label=f"DNN ({count_params(dnn):,} params)")
ax.plot(x, cnn_shift_acc, "s-", label=f"CNN ({count_params(cnn):,} params)")
ax.plot(x, cnn_gap_shift_acc, "^-", label=f"CNN + GAP ({count_params(cnn_gap):,} params)")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in shifts])
ax.set_xlabel("pixel shift (dx, dy)")
ax.set_ylabel("test accuracy")
ax.set_title("More built-in translation invariance = more robust to shift\n(at some cost in peak accuracy)")
ax.legend()
plt.tight_layout()
plt.show()

### 7. Inside the network: visualizing what each model learned

A DNN's first layer is simply a flat list of weights for each pixel. We can reshape
each hidden unit's weights back into a 28x28 image, but there is no reason for the
result to resemble anything meaningful, since the network never had a notion of
neighboring pixels.

A CNN's first layer filters, in contrast, are small (3x3) and are reused at every
position. As a result, they tend to converge into visually recognizable edge and blob
detectors.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
dnn_w1 = dnn.net[1].weight.detach()  # (256, 784)
for i, ax in enumerate(axes.flat):
    ax.imshow(dnn_w1[i].view(28, 28), cmap="gray")
    ax.axis("off")
fig.suptitle("DNN: 16 of 256 first-layer weight vectors, reshaped to 28x28 — look like noise")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
cnn_w1 = cnn.conv[0].weight.detach()  # (16, 1, 3, 3)
for i, ax in enumerate(axes.flat):
    ax.imshow(cnn_w1[i, 0], cmap="gray")
    ax.axis("off")
fig.suptitle("CNN: all 16 first-layer 3x3 filters — small, structured, reused everywhere")
plt.tight_layout()
plt.show()

In [ ]:
img, label = test_ds[1]
with torch.no_grad():
    feature_maps = cnn.conv[0:2](img.unsqueeze(0))  # conv1 + ReLU, before pooling

fig, axes = plt.subplots(1, 9, figsize=(14, 2.2))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"input\n({class_names[label]})", fontsize=8)
axes[0].axis("off")
for i in range(8):
    axes[i + 1].imshow(feature_maps[0, i].detach(), cmap="viridis")
    axes[i + 1].set_title(f"filter {i}", fontsize=8)
    axes[i + 1].axis("off")
fig.suptitle("What the first conv layer 'sees': each filter highlights a different local pattern")
plt.tight_layout()
plt.show()

### 8. Summary

| Limitation of a plain DNN on images | How a CNN addresses it |
| --- | --- |
| Flattening destroys the 2D neighborhood structure of pixels. | Convolutions operate directly on the 2D grid, so each output looks at a local neighborhood. |
| Every pixel and hidden unit pair has its own weight, so the parameter count grows with image size. | A kernel's weights are shared across every position, so the parameter count does not depend on image size. |
| There is no translation invariance. Shifting the object confuses the model, as measured above. | Weight sharing, together with pooling and global average pooling, lets the same filter recognize a feature wherever it appears. |
| First layer weights are not interpretable; they resemble pixel shaped noise. | Learned filters are small and visually interpretable, such as edges, blobs, and textures. |
| The network must relearn spatial structure from data, with no shortcuts. | Locality and weight sharing form a built in inductive bias suited to images, improving sample and parameter efficiency. |

Results from this run:

- Original test set: the DNN reached 87.3% accuracy with 235,146 parameters, while the
  CNN reached 89.5% accuracy with 206,922 parameters. The CNN achieved higher accuracy
  with fewer parameters.
- Shifted by two pixels: DNN accuracy fell to 27.4%, while CNN accuracy remained at
  65.8%. The convolutional feature extractor continued to recognize the same shapes.
- Shifted by four to six pixels, a large fraction of a 28x28 image: both the DNN
  (about 12%) and the plain CNN (about 15 to 16%) degraded toward chance level. The
  CNN with global average pooling, which discards positional information entirely
  before classification, retained 26% to 33% accuracy, making it the most robust of
  the three models.

These results are not due to any special property of CNNs. They follow directly and
predictably from local receptive fields, weight sharing, and, in the case of global
average pooling, the removal of position information.